# RANSAC multi-view hand keypoint triangulation (no WiLoR/MANO)

Pure-geometric companion to `wilor_hand_reconstruction_v2.ipynb` (left untouched).
Instead of fitting a MANO mesh per camera, this notebook:

1. Runs the same robust per-camera detection pipeline validated in
   `/tmp/diag_percam7_robust_render.py` (YOLO + MediaPipe detection union, temporal
   tracking, majority-vote handedness resolution, two-signal occlusion flagging) to
   get, for each of the 7 cameras and each frame, MediaPipe's 21 2D hand landmarks
   per hand (when available), plus presence/occlusion flags.
2. Triangulates each of the 21 landmarks into 3D using the calibrated camera
   projection matrices, with a RANSAC step (exhaustive camera pairs, reprojection-
   error inlier counting on the wrist joint) that automatically drops views where a
   hand is occluded or mislabeled.
3. Visualizes the resulting 3D hand skeletons in viser.

Runs entirely in `oak_env` (now includes `ultralytics` for YOLO).

In [ ]:
import sys, os, json, time, types, pickle
from itertools import combinations
from collections import Counter
import numpy as np
import cv2
import h5py
import torch
from ultralytics import YOLO

WILOR_DIR = '/home/fmalmb/CODE/WiLoR'
H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'
IMG_DIR = '/tmp/wilor_demo_input'  # produced by /tmp/extract_remaining_cams.py etc.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

detector = YOLO(os.path.join(WILOR_DIR, 'pretrained_models/detector.pt'))
detector = detector.to(DEVICE)
print(f'YOLO detector loaded on {DEVICE}')

In [ ]:
# Load Chameleon calibration from H5 (same approach as h5_explore.ipynb).
class ChameleonCalibrationV1:
    """Stub for unpickling when calib package is not installed."""
    pass


calib_mod = types.ModuleType('calib.chameleon_calibration')
calib_mod.ChameleonCalibrationV1 = ChameleonCalibrationV1
sys.modules['calib.chameleon_calibration'] = calib_mod
sys.modules['calib'] = types.ModuleType('calib')


def build_camera_cal(intrin_vec, cam_to_world):
    cx, cy, fx, fy = intrin_vec[2], intrin_vec[3], intrin_vec[4], intrin_vec[5]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    dist = intrin_vec[6:14].astype(np.float64)
    return K, dist


with h5py.File(H5_PATH, 'r') as f:
    cal = pickle.loads(bytes(f['calibration/cal_pickle'][()]))
    camera_ids = sorted(f['rec'].keys())

CAMERA_CALIB = {}
CAMERA_WORLD_TRANSFORMS = {}
PROJECTION_MATRICES = {}
for i, cam_id in enumerate(camera_ids):
    K, dist = build_camera_cal(cal.intrinsics[i], cal.cam_to_world[i])
    w, h = cal.intrinsics[i][0], cal.intrinsics[i][1]
    CAMERA_CALIB[cam_id] = {'K': K, 'dist': dist, 'width': float(w), 'height': float(h)}
    T_cw = cal.cam_to_world[i].astype(np.float64)
    CAMERA_WORLD_TRANSFORMS[cam_id] = T_cw
    # Maps a world point to this camera's 2D pixel (dist=None convention): P @ [X,Y,Z,1]
    # ~ K @ (R_wc @ X + t_wc). Pixel coordinates fed into this matrix must therefore be
    # undistorted first (see Stage 2).
    T_wc = np.linalg.inv(T_cw)
    PROJECTION_MATRICES[cam_id] = K @ T_wc[:3, :]
    print(f"  {cam_id}: fx={K[0, 0]:.1f}, fy={K[1, 1]:.1f}, cx={K[0, 2]:.1f}, cy={K[1, 2]:.1f}")

N_CAMS = len(camera_ids)
print(f'Loaded ChameleonCalibrationV1 ({cal.n_cameras} cameras, gravity={cal.gravity_world})')
print(f'{N_CAMS} cameras: {camera_ids}')

In [ ]:
with open('extracted_2d_keypoints.json') as f:
    keypoints_2d = json.load(f)

HAND_SIDES = ['left_hand', 'right_hand']
SIDE_IS_RIGHT = {'left_hand': 0.0, 'right_hand': 1.0}


def hand_bbox(landmarks):
    xs = np.array([lm['x'] for lm in landmarks], dtype=np.float32)
    ys = np.array([lm['y'] for lm in landmarks], dtype=np.float32)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)


def hand_boxes_for(entry):
    """Return (boxes, rights, sides) for the non-empty hand detections in a cam/frame entry."""
    boxes, rights, sides = [], [], []
    for side in HAND_SIDES:
        lms = entry[side]
        if len(lms) == 21:
            boxes.append(hand_bbox(lms))
            rights.append(SIDE_IS_RIGHT[side])
            sides.append(side)
    return boxes, rights, sides


def landmarks_array(landmarks):
    """21 MediaPipe landmarks -> (xy (21,2) float32, conf (21,) float32), sorted by id."""
    lms = sorted(landmarks, key=lambda lm: lm['id'])
    xy = np.array([[lm['x'], lm['y']] for lm in lms], dtype=np.float32)
    conf = np.array([lm['confidence'] for lm in lms], dtype=np.float32)
    return xy, conf

In [ ]:
# --- Stage 1: per-camera detection union, tracking, occlusion flagging --------------
# Ported from /tmp/diag_percam7_robust_render.py (validated this session), minus the
# WiLoR model/renderer -- only the YOLO detector is needed. Extended so each
# detection that has a MediaPipe match also carries that hand's 21 landmarks
# (`mp_landmarks`), which is the actual 2D input to triangulation below.

FRAME_OFFSET = 40
N_FRAMES = 50

IOU_THRESH = 0.2
MAX_GAP = 20
DIST_THRESH = 500.0
OCCLUSION_IOU_THRESH = 0.1
OCCLUSION_CONF = 0.5  # vote-weight / triangulation-weight multiplier for occlusion-flagged frames

N_JOINTS = 21


def yolo_boxes_for(img_cv2):
    detections = detector(img_cv2, conf=0.3, verbose=False)[0]
    boxes, rights = [], []
    for det in detections:
        Bbox = det.boxes.data.cpu().detach().squeeze().numpy()
        rights.append(det.boxes.cls.cpu().detach().squeeze().item())
        boxes.append(Bbox[:4].tolist())
    return boxes, rights


def box_center(box):
    return np.array([(box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0], dtype=np.float32)


def box_area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def iou(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    denom = box_area(a) + box_area(b) - inter
    return inter / denom if denom > 0 else 0.0


def union_detections(boxes_y, rights_y, boxes_m, rights_m, sides_m, entry):
    n_y, n_m = len(boxes_y), len(boxes_m)
    pairs = []
    for yi in range(n_y):
        for mi in range(n_m):
            v = iou(boxes_y[yi], boxes_m[mi])
            if v > IOU_THRESH:
                pairs.append((v, yi, mi))
    pairs.sort(key=lambda x: -x[0])

    used_y, used_m = set(), set()
    detections = []
    for v, yi, mi in pairs:
        if yi in used_y or mi in used_m:
            continue
        used_y.add(yi)
        used_m.add(mi)
        detections.append({
            'box': np.array(boxes_y[yi], dtype=np.float32),
            'votes': [(rights_y[yi], 'Y'), (rights_m[mi], 'M')],
            'sources': 'YM',
            'mp_landmarks': landmarks_array(entry[sides_m[mi]]),
        })
    for yi in range(n_y):
        if yi not in used_y:
            detections.append({
                'box': np.array(boxes_y[yi], dtype=np.float32),
                'votes': [(rights_y[yi], 'Y')],
                'sources': 'Y',
            })
    for mi in range(n_m):
        if mi not in used_m:
            detections.append({
                'box': np.array(boxes_m[mi], dtype=np.float32),
                'votes': [(rights_m[mi], 'M')],
                'sources': 'M',
                'mp_landmarks': landmarks_array(entry[sides_m[mi]]),
            })
    return detections


def build_tracks(per_frame_detections, n_frames):
    tracks = []  # {'frames': {fi: di}, 'last_center': np.array, 'last_fi': int}
    for fi in range(n_frames):
        dets = per_frame_detections[fi]
        centers = [box_center(d['box']) for d in dets]
        alive_idx = [ti for ti, t in enumerate(tracks) if fi - t['last_fi'] <= MAX_GAP]

        pairs = []
        for di, c in enumerate(centers):
            for ti in alive_idx:
                d = float(np.linalg.norm(c - tracks[ti]['last_center']))
                if d < DIST_THRESH:
                    pairs.append((d, di, ti))
        pairs.sort(key=lambda x: x[0])

        used_d, used_t = set(), set()
        for d, di, ti in pairs:
            if di in used_d or ti in used_t:
                continue
            used_d.add(di)
            used_t.add(ti)
            tracks[ti]['frames'][fi] = di
            tracks[ti]['last_center'] = centers[di]
            tracks[ti]['last_fi'] = fi

        for di in range(len(dets)):
            if di not in used_d:
                tracks.append({'frames': {fi: di}, 'last_center': centers[di], 'last_fi': fi})
    return tracks


def nearest_box(track, fi, per_frame_detections, n_frames):
    """Look forward and backward from frame `fi` for the temporally-nearest frame
    where `track` has a detection, and return that detection's box."""
    for d in range(1, n_frames):
        for f2 in (fi - d, fi + d):
            if f2 in track['frames']:
                return per_frame_detections[f2][track['frames'][f2]]['box']
    return None


def flag_occlusions(kept, per_frame_detections, n_frames):
    """For frames where exactly one of the two kept tracks has a detection and the
    other is absent (gap), the hands are likely close together / passing in front
    of each other. Flag the present detection as occlusion-ambiguous if either:
      1. its box overlaps the absent hand's nearest-in-time box (bb closeness), or
      2. it's a single merged YOLO+MediaPipe detection where the two detectors
         disagree on handedness (itself evidence the box covers both hands)."""
    if len(kept) != 2:
        return
    for fi in range(n_frames):
        present = [i for i, t in enumerate(kept) if fi in t['frames']]
        absent = [i for i, t in enumerate(kept) if fi not in t['frames']]
        if len(present) == 1 and len(absent) == 1:
            pi, ai = present[0], absent[0]
            det = per_frame_detections[fi][kept[pi]['frames'][fi]]

            other_box = nearest_box(kept[ai], fi, per_frame_detections, n_frames)
            iou_flag = other_box is not None and iou(det['box'], other_box) > OCCLUSION_IOU_THRESH

            disagree_flag = det['sources'] == 'YM' and det['votes'][0][0] != det['votes'][1][0]

            if iou_flag or disagree_flag:
                det['occlusion'] = True


def resolve_handedness(track, per_frame_detections):
    total = Counter()
    mp_only = Counter()
    for fi, di in track['frames'].items():
        det = per_frame_detections[fi][di]
        weight = OCCLUSION_CONF if det.get('occlusion') else 1.0
        for value, src in det['votes']:
            total[value] += weight
            if src == 'M':
                mp_only[value] += weight
    if total[1.0] != total[0.0]:
        return 1.0 if total[1.0] > total[0.0] else 0.0
    if mp_only[1.0] != mp_only[0.0]:
        return 1.0 if mp_only[1.0] > mp_only[0.0] else 0.0
    return 1.0


# hand_idx: 0 = left, 1 = right (global hand identity -- see notes_on_wilor_implementation.md)
keypoints_px = np.zeros((N_CAMS, N_FRAMES, 2, N_JOINTS, 2), dtype=np.float32)
keypoints_conf = np.zeros((N_CAMS, N_FRAMES, 2, N_JOINTS), dtype=np.float32)
presence = np.zeros((N_CAMS, N_FRAMES, 2), dtype=bool)
occlusion = np.zeros((N_CAMS, N_FRAMES, 2), dtype=bool)

for ci, cam_id in enumerate(camera_ids):
    t0 = time.time()
    per_frame_detections = []
    for fi in range(N_FRAMES):
        idx = FRAME_OFFSET + fi
        img = cv2.imread(f'{IMG_DIR}/{cam_id}/frame_{idx:06d}.png')
        boxes_y, rights_y = yolo_boxes_for(img)
        if len(boxes_y) > 2:
            order = sorted(range(len(boxes_y)), key=lambda i: -box_area(boxes_y[i]))[:2]
            boxes_y = [boxes_y[i] for i in order]
            rights_y = [rights_y[i] for i in order]

        frame_key = f'{idx:06d}'
        entry = keypoints_2d[frame_key][cam_id]
        boxes_m, rights_m, sides_m = hand_boxes_for(entry)

        dets = union_detections(boxes_y, rights_y, boxes_m, rights_m, sides_m, entry)
        per_frame_detections.append(dets)

    tracks = build_tracks(per_frame_detections, N_FRAMES)
    tracks.sort(key=lambda t: -len(t['frames']))
    kept = tracks[:2]

    flag_occlusions(kept, per_frame_detections, N_FRAMES)

    for t in kept:
        t['resolved'] = resolve_handedness(t, per_frame_detections)

    print(f'=== {cam_id} ({time.time() - t0:.1f}s, tracks found={len(tracks)}, kept={len(kept)}) ===')
    used_hand_idx = set()
    for t in kept:
        hand_idx = 1 if t['resolved'] > 0.5 else 0
        if hand_idx in used_hand_idx:
            print(f'  WARNING: both kept tracks resolved to hand_idx={hand_idx}; skipping second track')
            continue
        used_hand_idx.add(hand_idx)

        n_present = n_mp = n_occ = 0
        for fi, di in t['frames'].items():
            det = per_frame_detections[fi][di]
            n_present += 1
            is_occluded = bool(det.get('occlusion'))
            if is_occluded:
                n_occ += 1
            if 'mp_landmarks' in det:
                n_mp += 1
                xy, conf = det['mp_landmarks']
                keypoints_px[ci, fi, hand_idx] = xy
                keypoints_conf[ci, fi, hand_idx] = conf
                presence[ci, fi, hand_idx] = True
                occlusion[ci, fi, hand_idx] = is_occluded

        label = 'L' if hand_idx == 0 else 'R'
        print(f'  hand={label}: present {n_present}/{N_FRAMES}, mp_landmarks {n_mp}/{N_FRAMES}, '
              f'occlusion-flagged {n_occ}')

np.savez_compressed(
    'ransac_keypoint_detections.npz',
    keypoints_px=keypoints_px, keypoints_conf=keypoints_conf,
    presence=presence, occlusion=occlusion,
    camera_ids=np.array(camera_ids), frame_indices=np.arange(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES),
)
print('Saved ransac_keypoint_detections.npz')

In [ ]:
# --- Stage 2: RANSAC multi-view triangulation ----------------------------------------
RANSAC_REPROJ_THRESH_PX = 50.0  # wrist reprojection-error inlier threshold; tune from the
                                 # residual stats printed below (calibration baseline is
                                 # several tens of px even for "correct" triangulations).
RANSAC_HYSTERESIS_INLIER_MARGIN = 0   # keep the previous frame's inlier camera set unless a new
                                       # set beats it by more than this many inliers
RANSAC_HYSTERESIS_ERR_FACTOR = 1.5    # ...and its mean reprojection error isn't worse by more
                                       # than this factor


def triangulate_point_dlt_weighted(Ps, pixels, weights):
    """Weighted multi-view DLT triangulation (SVD), each pair of equations scaled by
    sqrt(weight)."""
    A = []
    for P, (x, y), wgt in zip(Ps, pixels, weights):
        sw = np.sqrt(wgt)
        A.append(sw * (x * P[2, :] - P[0, :]))
        A.append(sw * (y * P[2, :] - P[1, :]))
    A = np.array(A)
    _, _, Vh = np.linalg.svd(A)
    X_h = Vh[-1]
    return X_h[:3] / X_h[3]


def reproj_error_px(X, P, pixel):
    proj = P @ np.append(X, 1.0)
    return np.linalg.norm(proj[:2] / proj[2] - np.asarray(pixel))


# Undistort all keypoints up front (vectorized per camera) -- PROJECTION_MATRICES
# assumes the pinhole (dist=None) convention, so MediaPipe's raw/distorted pixel
# coords must be undistorted first.
keypoints_px_undist = np.zeros_like(keypoints_px)
for ci, cam_id in enumerate(camera_ids):
    K, dist = CAMERA_CALIB[cam_id]['K'], CAMERA_CALIB[cam_id]['dist']
    pts = keypoints_px[ci].reshape(-1, 1, 2).astype(np.float64)
    undist = cv2.undistortPoints(pts, K, dist, P=K)
    keypoints_px_undist[ci] = undist.reshape(N_FRAMES, 2, N_JOINTS, 2)

joints3d = np.full((N_FRAMES, 2, N_JOINTS, 3), np.nan, dtype=np.float64)
valid = np.zeros((N_FRAMES, 2), dtype=bool)
n_inliers = np.zeros((N_FRAMES, 2), dtype=np.int32)
reproj_err = np.full((N_FRAMES, 2, N_JOINTS), np.nan, dtype=np.float64)
wrist_residuals = []
hysteresis_overrides = [0, 0]
n_valid_frames = [0, 0]
prev_inliers = [None, None]

for fi in range(N_FRAMES):
    for hi in range(2):
        cams = np.where(presence[:, fi, hi])[0]
        if len(cams) < 2:
            continue

        # RANSAC on the wrist joint (landmark 0): try every camera pair, count
        # reprojection-error inliers across all candidate cameras, keep the pair
        # giving the most inliers (tie-break: lowest mean inlier residual).
        cand_inliers, cand_err = None, None
        for ca, cb in combinations(cams, 2):
            Ps_pair = [PROJECTION_MATRICES[camera_ids[c]] for c in (ca, cb)]
            px_pair = [keypoints_px_undist[c, fi, hi, 0] for c in (ca, cb)]
            X = triangulate_point_dlt_weighted(Ps_pair, px_pair, [1.0, 1.0])
            errs = np.array([
                reproj_error_px(X, PROJECTION_MATRICES[camera_ids[c]], keypoints_px_undist[c, fi, hi, 0])
                for c in cams
            ])
            inlier_mask = errs < RANSAC_REPROJ_THRESH_PX
            if inlier_mask.sum() < 2:
                continue
            mean_err = errs[inlier_mask].mean()
            inliers = cams[inlier_mask]
            if (cand_inliers is None or len(inliers) > len(cand_inliers)
                    or (len(inliers) == len(cand_inliers) and mean_err < cand_err)):
                cand_inliers, cand_err = inliers, mean_err

        if cand_inliers is None:
            continue

        best_inliers, best_err = cand_inliers, cand_err

        # Hysteresis: if the previous frame's chosen camera set is still (mostly)
        # present and reprojects acceptably here too, keep it instead of switching --
        # this damps the cm-scale 3D position jumps that occur when RANSAC's
        # per-frame-independent winner flips between similarly-good camera pairs.
        if prev_inliers[hi] is not None:
            prev_cams = np.array([c for c in prev_inliers[hi] if c in cams])
            if len(prev_cams) >= 2:
                Ps_prev = [PROJECTION_MATRICES[camera_ids[c]] for c in prev_cams]
                px_prev = [keypoints_px_undist[c, fi, hi, 0] for c in prev_cams]
                X_prev = triangulate_point_dlt_weighted(Ps_prev, px_prev, [1.0] * len(prev_cams))
                errs_prev = np.array([
                    reproj_error_px(X_prev, PROJECTION_MATRICES[camera_ids[c]], keypoints_px_undist[c, fi, hi, 0])
                    for c in cams
                ])
                prev_mask = errs_prev < RANSAC_REPROJ_THRESH_PX
                prev_count = int(prev_mask.sum())
                if prev_count >= 2:
                    prev_err = errs_prev[prev_mask].mean()
                    if (prev_count >= len(cand_inliers) - RANSAC_HYSTERESIS_INLIER_MARGIN
                            and prev_err <= cand_err * RANSAC_HYSTERESIS_ERR_FACTOR):
                        best_inliers, best_err = cams[prev_mask], prev_err

        if set(best_inliers.tolist()) != set(cand_inliers.tolist()):
            hysteresis_overrides[hi] += 1
        n_valid_frames[hi] += 1
        prev_inliers[hi] = tuple(best_inliers.tolist())

        valid[fi, hi] = True
        n_inliers[fi, hi] = len(best_inliers)
        wrist_residuals.append(best_err)

        for ji in range(N_JOINTS):
            Ps_j, px_j, w_j = [], [], []
            for c in best_inliers:
                Ps_j.append(PROJECTION_MATRICES[camera_ids[c]])
                px_j.append(keypoints_px_undist[c, fi, hi, ji])
                w = max(float(keypoints_conf[c, fi, hi, ji]), 1e-3)
                if occlusion[c, fi, hi]:
                    w *= OCCLUSION_CONF
                w_j.append(w)
            X = triangulate_point_dlt_weighted(Ps_j, px_j, w_j)
            joints3d[fi, hi, ji] = X
            reproj_err[fi, hi, ji] = np.mean([reproj_error_px(X, P, px) for P, px in zip(Ps_j, px_j)])

for hi, label in enumerate(['left', 'right']):
    n_valid = int(valid[:, hi].sum())
    print(f'{label} hand: {n_valid}/{N_FRAMES} frames valid')
    if n_valid:
        ni = n_inliers[valid[:, hi], hi]
        print(f'  inlier cams: min={ni.min()}, mean={ni.mean():.2f}, max={ni.max()}')
        print(f'  mean reprojection error: {np.nanmean(reproj_err[valid[:, hi], hi, :]):.1f}px')
        print(f'  hysteresis kept previous camera set: {hysteresis_overrides[hi]}/{n_valid_frames[hi]} valid frames')

wrist_residuals = np.array(wrist_residuals)
print(f'wrist RANSAC residual (px): mean={wrist_residuals.mean():.1f}, '
      f'median={np.median(wrist_residuals):.1f}, max={wrist_residuals.max():.1f}')

In [ ]:
# --- Stage 2c: Kalman/RTS smoothing of joint trajectories -----------------------------
# Constant-velocity Kalman filter + Rauch-Tung-Striebel (RTS) backward smoother, applied
# independently to each joint's 3D trajectory. Frames with no RANSAC solution (valid=False)
# get pure predictions, filled in by the backward pass using later valid frames.
# Measurement variance is inflated for n_inliers==2 triangulations, which have larger depth
# uncertainty (dZ ~ Z^2 * dpx / (fx * B), see analyze_camera_jitter.py).
KALMAN_PROCESS_STD = 0.03     # m / frame^2 acceleration-noise std (raise if smoothing lags fast motion)
KALMAN_MEAS_STD_BASE = 0.01   # m position-measurement std for an n_inliers>=3 triangulation
KALMAN_MEAS_STD_2CAM = 0.015  # m position-measurement std for an n_inliers==2 triangulation


def kalman_rts_smooth(positions, valid_mask, meas_std):
    """positions: (T,3); valid_mask: (T,) bool; meas_std: (T,) per-frame std (m).
    Forward constant-velocity Kalman filter + RTS backward pass. Returns (T,3) smoothed
    positions, or all-NaN if valid_mask is all False."""
    T = positions.shape[0]
    if not valid_mask.any():
        return np.full((T, 3), np.nan)

    F = np.eye(6)
    F[:3, 3:] = np.eye(3)
    q = KALMAN_PROCESS_STD ** 2
    Q = np.block([
        [np.eye(3) * (q / 4.0), np.eye(3) * (q / 2.0)],
        [np.eye(3) * (q / 2.0), np.eye(3) * q],
    ])
    H = np.hstack([np.eye(3), np.zeros((3, 3))])

    first = int(np.argmax(valid_mask))
    x = np.zeros(6)
    x[:3] = positions[first]
    P = np.eye(6) * 1.0

    xs_pred = np.zeros((T, 6))
    Ps_pred = np.zeros((T, 6, 6))
    xs_filt = np.zeros((T, 6))
    Ps_filt = np.zeros((T, 6, 6))

    for t in range(T):
        if t > 0:
            x = F @ x
            P = F @ P @ F.T + Q
        xs_pred[t], Ps_pred[t] = x, P
        if valid_mask[t]:
            R = np.eye(3) * meas_std[t] ** 2
            S = H @ P @ H.T + R
            K = P @ H.T @ np.linalg.inv(S)
            x = x + K @ (positions[t] - H @ x)
            P = (np.eye(6) - K @ H) @ P
        xs_filt[t], Ps_filt[t] = x, P

    xs_smooth = xs_filt.copy()
    for t in range(T - 2, -1, -1):
        C = Ps_filt[t] @ F.T @ np.linalg.inv(Ps_pred[t + 1])
        xs_smooth[t] = xs_filt[t] + C @ (xs_smooth[t + 1] - xs_pred[t + 1])

    return xs_smooth[:, :3]


joints3d_smooth = np.full_like(joints3d, np.nan)
valid_smooth = np.zeros((N_FRAMES, 2), dtype=bool)
for hi in range(2):
    valid_idx = np.where(valid[:, hi])[0]
    if len(valid_idx) == 0:
        continue
    valid_smooth[valid_idx[0]:valid_idx[-1] + 1, hi] = True
    meas_std = np.where(n_inliers[:, hi] >= 3, KALMAN_MEAS_STD_BASE, KALMAN_MEAS_STD_2CAM)
    for ji in range(N_JOINTS):
        joints3d_smooth[:, hi, ji] = kalman_rts_smooth(joints3d[:, hi, ji], valid[:, hi], meas_std)

for hi, label in enumerate(['left', 'right']):
    vi = valid[:, hi]
    if not vi.any():
        continue
    delta = np.linalg.norm(joints3d[vi, hi, 0] - joints3d_smooth[vi, hi, 0], axis=-1)
    print(f'{label} hand wrist: raw-vs-smoothed correction (cm): '
          f'mean={delta.mean() * 100:.2f}, median={np.median(delta) * 100:.2f}, max={delta.max() * 100:.2f}')

In [ ]:
# --- Stage 2d: jitter flagging (raw RANSAC vs Kalman/RTS-smoothed wrist) ---------------
# Frames where the raw RANSAC wrist position deviates sharply from the Kalman/RTS-smoothed
# trajectory are flagged -- a sign that frame's RANSAC camera-set choice was an outlier,
# now corrected by the smoother.
JITTER_MAD_FACTOR = 4.0  # flag residuals > median + this many MADs above it

jitter_flag = np.zeros((N_FRAMES, 2), dtype=bool)
wrist_jitter_residual = np.full((N_FRAMES, 2), np.nan)

for hi, label in enumerate(['left', 'right']):
    valid_idx = np.where(valid[:, hi])[0]
    if len(valid_idx) == 0:
        continue
    residual = np.linalg.norm(joints3d[valid_idx, hi, 0] - joints3d_smooth[valid_idx, hi, 0], axis=-1)
    wrist_jitter_residual[valid_idx, hi] = residual

    med = np.median(residual)
    mad = np.median(np.abs(residual - med)) + 1e-9
    thresh = med + JITTER_MAD_FACTOR * mad
    flagged = valid_idx[residual > thresh]
    jitter_flag[flagged, hi] = True

    print(f'{label} hand wrist raw-vs-smoothed residual (cm): median={med * 100:.2f}, '
          f'mad={mad * 100:.2f}, threshold={thresh * 100:.2f}, max={residual.max() * 100:.2f}')
    print(f'  flagged frames ({len(flagged)}/{len(valid_idx)}, global idx): '
          f'{(flagged + FRAME_OFFSET).tolist()}')

In [ ]:
# --- Stage 2e: Savitzky-Golay smoothing + 2x frame-rate interpolation -----------------
# Apply a Savitzky-Golay filter (local polynomial fit) to the Kalman/RTS-smoothed
# trajectories for additional noise reduction without the lag a low-pass filter would add,
# then upsample to 2x the original frame rate via cubic-spline interpolation of the now-smooth
# curves -- doubling the temporal resolution of the 3D reconstruction.
from scipy.signal import savgol_filter
from scipy.interpolate import CubicSpline

SG_WINDOW = 7  # frames (must be odd and <= the valid span length)
SG_POLYORDER = 3

joints3d_sg = np.full_like(joints3d_smooth, np.nan)
for hi in range(2):
    valid_idx = np.where(valid_smooth[:, hi])[0]
    if len(valid_idx) == 0:
        continue
    lo, hi_end = valid_idx[0], valid_idx[-1] + 1
    span = hi_end - lo
    window = min(SG_WINDOW, span if span % 2 == 1 else span - 1)
    if window <= SG_POLYORDER:
        joints3d_sg[lo:hi_end, hi] = joints3d_smooth[lo:hi_end, hi]
        continue
    for ji in range(N_JOINTS):
        joints3d_sg[lo:hi_end, hi, ji] = savgol_filter(
            joints3d_smooth[lo:hi_end, hi, ji], window_length=window, polyorder=SG_POLYORDER, axis=0)

# Upsample to 2x frame rate: cubic-spline-interpolate the SG-smoothed trajectories at the
# half-integer frame positions (0, 0.5, 1, ..., N_FRAMES-1) -> 2*N_FRAMES - 1 samples.
N_FRAMES_2X = 2 * N_FRAMES - 1
t_orig = np.arange(N_FRAMES, dtype=np.float64)
t_2x = np.arange(N_FRAMES_2X, dtype=np.float64) * 0.5
nearest_orig_2x = np.clip(np.round(t_2x).astype(np.int64), 0, N_FRAMES - 1)

joints3d_2x = np.full((N_FRAMES_2X, 2, N_JOINTS, 3), np.nan, dtype=np.float64)
valid_2x = np.zeros((N_FRAMES_2X, 2), dtype=bool)
for hi in range(2):
    valid_idx = np.where(valid_smooth[:, hi])[0]
    if len(valid_idx) == 0:
        continue
    lo, hi_end = valid_idx[0], valid_idx[-1] + 1
    t_mask = (t_2x >= t_orig[lo]) & (t_2x <= t_orig[hi_end - 1])
    valid_2x[t_mask, hi] = True
    for ji in range(N_JOINTS):
        for d in range(3):
            cs = CubicSpline(t_orig[lo:hi_end], joints3d_sg[lo:hi_end, hi, ji, d])
            joints3d_2x[t_mask, hi, ji, d] = cs(t_2x[t_mask])

# Nearest-original-frame upsampling of per-frame diagnostics, for visualization at 2x rate.
jitter_flag_2x = jitter_flag[nearest_orig_2x]
n_inliers_2x = n_inliers[nearest_orig_2x]

for hi, label in enumerate(['left', 'right']):
    vi = valid_smooth[:, hi]
    if not vi.any():
        continue
    delta = np.linalg.norm(joints3d_smooth[vi, hi, 0] - joints3d_sg[vi, hi, 0], axis=-1)
    n2x = int(valid_2x[:, hi].sum())
    print(f'{label} hand: Savitzky-Golay correction on wrist (cm): mean={delta.mean() * 100:.3f}, '
          f'max={delta.max() * 100:.3f}; {n2x}/{N_FRAMES_2X} frames valid at 2x rate')

In [ ]:
# --- Stage 3: viser visualization -----------------------------------------------------
import viser
from viser import transforms as tf

# Calibration world is Y-up; Viser uses Z-up. Flip world Y so the figure is upright
# (same convention as wilor_hand_reconstruction_v2.ipynb).
R_WORLD_TO_VISER = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, -1.0, 0.0],
], dtype=np.float64)

HAND_NAMES = {0: 'left', 1: 'right'}
HAND_COLORS = {0: (180, 220, 255), 1: (255, 200, 170)}
LOW_CONFIDENCE_COLOR = (140, 140, 140)
JITTER_COLOR = (230, 40, 40)  # frames where raw RANSAC wrist deviated sharply from the
                               # Kalman/RTS-smoothed trajectory (jitter_flag above)

# Standard MediaPipe 21-point hand topology (thumb/index/middle/ring/pinky chains + palm).
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
]


def world_to_viser_point(p_world, origin_world):
    return R_WORLD_TO_VISER @ (np.asarray(p_world, dtype=np.float64) - origin_world)


def world_to_viser_pose(T_cw, origin_world):
    T = np.asarray(T_cw, dtype=np.float64).copy()
    T[:3, 3] = R_WORLD_TO_VISER @ (T[:3, 3] - origin_world)
    T[:3, :3] = R_WORLD_TO_VISER @ T[:3, :3]
    wxyz = tf.SO3.from_matrix(T[:3, :3]).wxyz
    return wxyz.astype(np.float64), T[:3, 3].astype(np.float64)


# Origin: centroid of all valid joints (both hands) in the first valid frame.
_origin_frame = 0
while _origin_frame < N_FRAMES and not valid[_origin_frame].any():
    _origin_frame += 1
if _origin_frame < N_FRAMES:
    _pts = np.concatenate(
        [joints3d[_origin_frame, hi] for hi in range(2) if valid[_origin_frame, hi]], axis=0)
    ORIGIN_WORLD = _pts.mean(axis=0)
else:
    ORIGIN_WORLD = np.zeros(3)


def setup_cameras(server):
    for cam_id, T_cw in CAMERA_WORLD_TRANSFORMS.items():
        calib = CAMERA_CALIB[cam_id]
        K = calib['K']
        fov = 2.0 * np.arctan(calib['height'] / (2.0 * K[1, 1]))
        aspect = float(calib['width'] / calib['height'])
        wxyz, position = world_to_viser_pose(T_cw, ORIGIN_WORLD)
        server.scene.add_camera_frustum(
            name=f'/cameras/{cam_id}', fov=float(fov), aspect=aspect, scale=0.1,
            line_width=0.5, color=(80, 160, 255), wxyz=wxyz, position=position,
        )


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.gui.configure_theme(dark_mode=True)
        setup_cameras(server)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        stop_button = server.gui.add_button('Stop viewer')
        frame_slider = server.gui.add_slider('Frame (2x rate)', min=0, max=N_FRAMES_2X - 1, step=1, initial_value=0)
        play_checkbox = server.gui.add_checkbox('Play (loop)', initial_value=True)
        fps_slider = server.gui.add_slider('Playback FPS', min=1, max=60, step=1, initial_value=20)

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        last_advance = time.time()

        while running:
            fi = frame_slider.value
            for hi in range(2):
                name = f'/hands/{HAND_NAMES[hi]}'
                if valid_2x[fi, hi]:
                    pts = np.stack([
                        world_to_viser_point(joints3d_2x[fi, hi, ji], ORIGIN_WORLD) for ji in range(N_JOINTS)
                    ]).astype(np.float32)
                    if jitter_flag_2x[fi, hi]:
                        color = JITTER_COLOR
                    elif n_inliers_2x[fi, hi] >= 3:
                        color = HAND_COLORS[hi]
                    else:
                        color = LOW_CONFIDENCE_COLOR
                    server.scene.add_point_cloud(
                        name=f'{name}/joints', points=pts,
                        colors=np.full((N_JOINTS, 3), color, dtype=np.uint8), point_size=0.008,
                    )
                    segs = np.array([[pts[a], pts[b]] for a, b in HAND_CONNECTIONS], dtype=np.float32)
                    server.scene.add_line_segments(
                        name=f'{name}/skeleton', points=segs,
                        colors=np.array(color, dtype=np.uint8), line_width=2.0,
                    )
                else:
                    server.scene.add_point_cloud(
                        name=f'{name}/joints', points=np.zeros((0, 3), dtype=np.float32),
                        colors=np.zeros((0, 3), dtype=np.uint8),
                    )
                    server.scene.add_line_segments(
                        name=f'{name}/skeleton', points=np.zeros((0, 2, 3), dtype=np.float32),
                        colors=np.zeros((0, 2, 3), dtype=np.uint8),
                    )

            if play_checkbox.value:
                now = time.time()
                if now - last_advance >= 1.0 / fps_slider.value:
                    frame_slider.value = (fi + 1) % N_FRAMES_2X
                    last_advance = now

            time.sleep(0.02)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()